# ClinVar SNV araştırma akışı — v2

Bu notebook çevrimdışı inceleme katmanıdır. Veri indirme, anotasyon ve eğitim işlemleri ortak `clinvar_classifier` paketinin CLI komutlarıyla yapılır. Eski çalışmadan kalan hücre çıktıları güncel sonuç gibi gösterilmez. Model skoru klinik patojenite olasılığı değildir.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT / 'src'))
from clinvar_classifier.data import map_label
from clinvar_classifier.features import make_features
from clinvar_classifier.inference import Predictor
BUNDLE = ROOT / 'artifacts' / 'research-v2'


## Açık etiket ve eksiklik politikası

P/LP ve B/LB etiketleri açık izin listesiyle eşlenir. Düşük penetrans, risk alleli ve karışık/çelişkili etiketler ikili hedefe dahil edilmez. Eksik AF ve gözlenen sıfır ayrı tutulur.


In [ ]:
labels = ['Pathogenic', 'Likely pathogenic', 'Pathogenic/Likely pathogenic', 'Benign', 'Likely benign', 'Benign/Likely benign', 'Uncertain significance', 'Pathogenic, low penetrance']
print(pd.DataFrame({'ClinicalSignificance': labels, 'label': [map_label(x) for x in labels]}).to_string(index=False))
raw = pd.DataFrame({'cadd_phred': [20.,20.], 'phylop_100way': [2.,2.], 'gnomad_af': [None,0.], 'af_available': [False,True]})
print(make_features(raw))


## Tekrarlanabilir tam koşu

Repo root'unda `scripts/download_snapshot.py` ile tarihli snapshot indirilir. Ardından `prepare`, `annotate`, `train`, `evaluate-external`, `score-vus` komutları kullanılır; ayrıntılar README'dedir.

Model seçimi yalnızca fit partition ve gen ayrık CV fold'larında yapılır. Kalibrasyon için farklı satırlar kullanılır. Varyant ve bağlı gen grubu testleri tüm seçimler bittikten sonra açılır. Ekim snapshot'ındaki yeni etiketli ve Temmuz VUS'tan değişen kayıtlar, Temmuz'daki tüm etiketli kohorttan kimlik bazında ayrılır.


In [ ]:
manifest = json.loads((BUNDLE / 'manifest.json').read_text(encoding='utf-8')) if (BUNDLE / 'manifest.json').exists() else None
if manifest:
    print({k: manifest.get(k) for k in ['run_id','n_fit','n_calibration','feature_contract','validation_status']})
    print('Training snapshot SHA256:', manifest.get('provenance', {}).get('cohort', {}).get('snapshot_sha256'))
else:
    print('No research-v2 bundle yet. Run the documented training commands first; no placeholder performance is displayed.')


## Kayıtlı değerlendirme ve ablation

Bu hücre sonuç dosyalarını okur; test verisiyle yeniden model veya özellik seçimi yapmaz. Benzer CV ortalamaları istatistiksel eşdeğerlik kanıtı sayılmaz.


In [ ]:
if manifest and (BUNDLE / 'metrics.json').exists():
    evaluation = json.loads((BUNDLE / 'metrics.json').read_text(encoding='utf-8'))
    summaries = []
    for name, result in evaluation['test'].items():
        v = result['calibrated_score']
        summaries.append({'partition':name,'n':v['n'],'ROC-AUC':v['roc_auc'],'Brier':v['brier'],'log_loss':v['log_loss']})
    print(pd.DataFrame(summaries).to_string(index=False))
    scores = pd.DataFrame(evaluation['development']['fold_scores'])
    print(scores.groupby('model').roc_auc.agg(['mean','std']))


## Bağımsız satırlarda kalibrasyon ve zaman kontrolü

Kalibrasyon, bu veri dağılımında skorları ayarlamaktır; hasta riski veya penetrans doğrulaması değildir. Zaman bazlı ClinVar etiket kontrolü bağımsız fonksiyonel deney yerine geçmez. Çıktıdaki alt gruplar ve kutu örneklem sayıları birlikte okunmalıdır.


In [ ]:
if manifest and (BUNDLE / 'external_metrics.json').exists():
    external = json.loads((BUNDLE / 'external_metrics.json').read_text(encoding='utf-8'))
    print(external['interpretation'])
    print(pd.DataFrame({name:{k:value.get(k) for k in ['n','n_benign','n_pathogenic','roc_auc','brier']} for name,value in external['groups'].items()}).T)


## Model davranışı ve kapsam

Tree SHAP raw log-odds marginini açıklar; klinik kanıt veya bağımsız nedensel açıklama değildir. HFE p.Cys282Tyr, düşük penetrans/sık patojenik allel sınırlamasının belgelenmiş örneğidir ve bilinen benign kontrol olarak sunulmaz.


In [ ]:
if manifest:
    predictor = Predictor(BUNDLE)
    example = pd.DataFrame([{'cadd_phred':25.,'phylop_100way':2.,'gnomad_af':None,'af_available':False}])
    print(predictor.predict(example))
    features, values, base = predictor.explain(example)
    print(pd.Series(values[0], index=features.columns, name='Tree SHAP raw-margin contribution'))
    print('Base margin:', base[0])
